# Day 6 – Feature Engineering

## 1. Title
Feature engineering techniques for preparing raw data for machine learning.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

df = pd.DataFrame({
    "Age": [22, 35, 47, np.nan, 29, 61],
    "Salary": [30000, 50000, 70000, 45000, np.nan, 120000],
    "Gender": ["Female", "Male", "Female", "Male", np.nan, "Female"],
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Pune"],
    "Date": ["2024-01-10", "2024-02-15", "2024-03-20",
             "2024-04-05", "2024-05-12", "2024-06-25"],
    "Purchase_Amount": [2000, 5000, 4500, 3000, 3500, 9000],
    "Experience": [1, 5, 10, 3, 4, 20],
    "Rating": [4.2, 4.5, 4.0, 3.8, 4.4, 4.9]
})

print("Original data:")
display(df)

### Problem 1: Handle missing values


for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].mean())

for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum())

### Problem 2: Scale Salary and Age

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

df["Salary_Standardized"] = standard_scaler.fit_transform(df[["Salary"]])
df["Age_MinMax"] = minmax_scaler.fit_transform(df[["Age"]])

display(df[["Salary", "Salary_Standardized", "Age", "Age_MinMax"]])

label_encoder = LabelEncoder()
df["Gender_Encoded"] = label_encoder.fit_transform(df["Gender"])

df = pd.get_dummies(df, columns=["City"], prefix="City", dtype=int)
display(df.head())


### Problems 4, 5, and 6: Create features, extract dates, and bin ages


df["Total_Spending"] = df["Salary"] + df["Purchase_Amount"]

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Weekday"] = df["Date"].dt.day_name()

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[-1, 25, 45, np.inf],
    labels=["Young", "Adult", "Senior"]
)


### Problem 7: Detect Salary outliers using IQR


q1 = df["Salary"].quantile(0.25)
q3 = df["Salary"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

salary_outliers = df[
    (df["Salary"] < lower_bound) |
    (df["Salary"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
display(salary_outliers)

### Problem 8: Create an interaction feature


df["Experience_Salary"] = df["Experience"] * df["Salary"]
display(df[["Experience", "Salary", "Experience_Salary"]])


### Problem 9: Correlation-based feature inspection

numeric_df = df.select_dtypes(include="number")
correlations = numeric_df.corr(numeric_only=True)["Purchase_Amount"].drop(
    labels=["Purchase_Amount"], errors="ignore"
)

print("Correlation with Purchase_Amount:")
print(correlations.sort_values(key=abs, ascending=False))

if not correlations.empty:
    least_correlated_feature = correlations.abs().idxmin()
    print("Least correlated feature:", least_correlated_feature)
    final_df = df.drop(columns=[least_correlated_feature])
else:
    final_df = df.copy()

display(final_df.head())


Original data:


,Age,Salary,Gender,City,Date,Purchase_Amount,Experience,Rating
0,22.0,30000.0,Female,Pune,2024-01-10,2000,1,4.2
1,35.0,50000.0,Male,Mumbai,2024-02-15,5000,5,4.5
2,47.0,70000.0,Female,Pune,2024-03-20,4500,10,4.0
3,NaN,45000.0,Male,Delhi,2024-04-05,3000,3,3.8
4,29.0,NaN,NaN,Mumbai,2024-05-12,3500,4,4.4
5,61.0,120000.0,Female,Pune,2024-06-25,9000,20,4.9


Missing values after imputation:
Age                0
Salary             0
Gender             0
City               0
Date               0
Purchase_Amount    0
Experience         0
Rating             0
dtype: int64


,Salary,Salary_Standardized,Age,Age_MinMax
0,30000.0,-1.157123,22.0,0.000000
1,50000.0,-0.455836,35.0,0.333333
2,70000.0,0.245450,47.0,0.641026
3,45000.0,-0.631158,38.8,0.430769
4,63000.0,0.000000,29.0,0.179487
5,120000.0,1.998668,61.0,1.000000


,Age,Salary,Gender,Date,Purchase_Amount,Experience,Rating,Salary_Standardized,Age_MinMax,Gender_Encoded,City_Delhi,City_Mumbai,City_Pune
0,22.0,30000.0,Female,2024-01-10,2000,1,4.2,-1.157123,0.000000,0,0,0,1
1,35.0,50000.0,Male,2024-02-15,5000,5,4.5,-0.455836,0.333333,1,0,1,0
2,47.0,70000.0,Female,2024-03-20,4500,10,4.0,0.245450,0.641026,0,0,0,1
3,38.8,45000.0,Male,2024-04-05,3000,3,3.8,-0.631158,0.430769,1,1,0,0
4,29.0,63000.0,Female,2024-05-12,3500,4,4.4,0.000000,0.179487,0,0,1,0


Lower bound: 13250.0
Upper bound: 101250.0


,Age,Salary,Gender,Date,Purchase_Amount,Experience,Rating,Salary_Standardized,Age_MinMax,Gender_Encoded,City_Delhi,City_Mumbai,City_Pune,Total_Spending,Year,Month,Day,Weekday,Age_Group
5,61.0,120000.0,Female,2024-06-25,9000,20,4.9,1.998668,1.0,0,0,0,1,129000.0,2024,6,25,Tuesday,Senior


,Experience,Salary,Experience_Salary
0,1,30000.0,30000.0
1,5,50000.0,250000.0
2,10,70000.0,700000.0
3,3,45000.0,135000.0
4,4,63000.0,252000.0
5,20,120000.0,2400000.0


Correlation with Purchase_Amount:
Experience             0.954867
Total_Spending         0.947832
Experience_Salary      0.945427
Salary                 0.939568
Salary_Standardized    0.939568
Age_MinMax             0.886034
Age                    0.886034
Day                    0.859009
Rating                 0.774880
Month                  0.632832
City_Delhi            -0.300000
City_Pune              0.298142
Gender_Encoded        -0.158114
City_Mumbai           -0.079057
Year                        NaN
Name: Purchase_Amount, dtype: float64
Least correlated feature: City_Mumbai


,Age,Salary,Gender,Date,Purchase_Amount,Experience,Rating,Salary_Standardized,Age_MinMax,Gender_Encoded,City_Delhi,City_Pune,Total_Spending,Year,Month,Day,Weekday,Age_Group,Experience_Salary
0,22.0,30000.0,Female,2024-01-10,2000,1,4.2,-1.157123,0.000000,0,0,1,32000.0,2024,1,10,Wednesday,Young,30000.0
1,35.0,50000.0,Male,2024-02-15,5000,5,4.5,-0.455836,0.333333,1,0,0,55000.0,2024,2,15,Thursday,Adult,250000.0
2,47.0,70000.0,Female,2024-03-20,4500,10,4.0,0.245450,0.641026,0,0,1,74500.0,2024,3,20,Wednesday,Senior,700000.0
3,38.8,45000.0,Male,2024-04-05,3000,3,3.8,-0.631158,0.430769,1,1,0,48000.0,2024,4,5,Friday,Adult,135000.0
4,29.0,63000.0,Female,2024-05-12,3500,4,4.4,0.000000,0.179487,0,0,0,66500.0,2024,5,12,Sunday,Adult,252000.0
